# 🛡️ CartShield AI — EDA & Modeling Notebook
**Return Fraud Detection with XGBoost + SHAP**

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
print('Libraries loaded ✅')

## 1. Load & Inspect Data

In [ ]:
df = pd.read_csv('../data/raw/transactions.csv')
print(f'Shape: {df.shape}')
print(f'Fraud rate: {df["is_fraud"].mean():.2%}')
print(f'Return rate: {df["has_return"].mean():.2%}')
df.head(3)

In [ ]:
df.info()
df.describe()

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Fraud by category
cat_fraud = df.groupby('product_category')['is_fraud'].mean().sort_values(ascending=False)
cat_fraud.plot(kind='bar', ax=axes[0], color='#e74c3c', edgecolor='white')
axes[0].set_title('Fraud Rate by Category', fontweight='bold')
axes[0].set_ylabel('Fraud Rate')
axes[0].tick_params(axis='x', rotation=30)

# Order value distribution
df['order_value'].clip(0, 1000).plot(kind='hist', bins=50, ax=axes[1], color='#3498db', edgecolor='white')
axes[1].set_title('Order Value Distribution', fontweight='bold')
axes[1].set_xlabel('Order Value ($)')

# Refund frequency
df['refund_frequency_30d'].value_counts().sort_index().plot(kind='bar', ax=axes[2], color='#9b59b6', edgecolor='white')
axes[2].set_title('Refund Frequency (30d)', fontweight='bold')
axes[2].set_xlabel('Number of Refunds')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (numeric only)
numeric_cols = ['order_value','refund_amount','delivery_days','days_to_return',
                'refund_frequency_30d','account_age_days','is_fraud']
corr = df[numeric_cols].corr()

plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Fraud rate by payment method
pay_fraud = df.groupby('payment_method')['is_fraud'].agg(['mean','count']).sort_values('mean', ascending=False)
pay_fraud.columns = ['Fraud Rate', 'Volume']
print(pay_fraud.to_string())

pay_fraud['Fraud Rate'].plot(kind='barh', color='#e67e22', figsize=(8,4))
plt.title('Fraud Rate by Payment Method', fontweight='bold')
plt.xlabel('Fraud Rate')
plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
from src.data.feature_engineering import FeatureEngineer

df_ret = df[df['has_return'] == 1].copy()
fe = FeatureEngineer()
df_feat = fe.fit_transform(df_ret)

print(f'Features engineered: {len(fe.get_feature_names())}')
print('\nNew features:')
for f in fe.get_feature_names():
    print(f'  → {f}')

## 4. Model Training

In [ ]:
from src.models.train import train
model, fe, metrics = train()
print('\nMetrics:', metrics)

## 5. SHAP Explainability

In [ ]:
import shap
from src.models.train import explain

feature_cols = fe.get_feature_names()
feature_cols = [c for c in feature_cols if c not in ['is_fraud','fraud_type','has_return']]
df_sample = fe.transform(df_ret.head(300))[feature_cols].fillna(0)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(df_sample)

shap.summary_plot(shap_values, df_sample, max_display=15)

In [ ]:
# Waterfall for a single high-risk transaction
idx = 5
shap.waterfall_plot(
    shap.Explanation(values=shap_values[idx], 
                     base_values=explainer.expected_value,
                     data=df_sample.iloc[idx].values,
                     feature_names=list(df_sample.columns))
)

## 6. Customer Segmentation

In [ ]:
from src.models.segmentation import segment_customers

profile = segment_customers(df)
print(profile.groupby('segment_label')[['total_orders','return_rate','fraud_rate']].mean().round(3))